# Consolidated Invoice Report — Agrosys (DW)

This notebook extracts data from the Agrosys ERP Data Warehouse via ODBC, consolidates the **invoice items, invoice header, product registry, customers, and cities** tables, and exports the final result to Excel.

---

## 1. Initial Setup


In [ ]:
import pandas as pd
import pyodbc
import warnings
import os
from dotenv import load_dotenv

# Suppress unnecessary warnings during execution
warnings.filterwarnings('ignore')

# Load environment variables from the .env file (not pushed to GitHub)
load_dotenv()

---

## 2. Database Connection and Table Extraction

The connection string uses a DSN (Data Source Name) configured in Windows ODBC.  
All tables are loaded sequentially, and the connection is closed immediately afterward.


In [ ]:
# ODBC connection string — points to the Agrosys Data Warehouse
dsn = os.getenv("AGROSYS_DSN")
user = os.getenv("AGROSYS_USER")
password = os.getenv("AGROSYS_PWD")

connection_string = f"DSN={dsn};UID={user};PWD={password};"

# Open the database connection
connection = pyodbc.connect(connection_string)

# Load the 5 required tables
print("Fetching dw_danfite...")
df_danfite = pd.read_sql('SELECT * FROM "pub"."dw_danfite"', connection)   # Invoice items

print("Fetching dw_danfe...")
df_danfe = pd.read_sql('SELECT * FROM "pub"."dw_danfe"', connection)       # Invoice header

print("Fetching dw_item...")
df_item = pd.read_sql('SELECT * FROM "pub"."dw_item"', connection)         # Product registry

print("Fetching dw_cliente...")
df_cliente = pd.read_sql('SELECT * FROM "pub"."dw_cliente"', connection)   # Customer registry

print("Fetching dw_cidade...")
df_cidade = pd.read_sql('SELECT * FROM "pub"."dw_cidade"', connection)     # City registry

# Close the connection immediately after loading
connection.close()
print("\nAll tables were loaded and the connection was safely closed!")

---

## 3. Inspecting Available Columns

Lists the columns of each table to make selection easier in the following steps.


In [ ]:
print("DANFITE columns:", list(df_danfite.columns))
print("\nDANFE columns:",   list(df_danfe.columns))
print("\nITEM columns:",    list(df_item.columns))
print("\nCLIENTE columns:", list(df_cliente.columns))
print("\nCIDADE columns:",  list(df_cidade.columns))

---

## 4. Selecting Relevant Columns

To avoid working with overly wide tables, only the necessary columns from each source are selected before performing the merges.


In [ ]:
# Selected columns from each table
selected_danfite_cols  = ['id_danfe_numint', 'id_empresa', 'descricao_item', 'id_item', 'volumes', 'id_cfop', 'valor_liquido']
selected_danfe_cols    = ['id_danfe_numint', 'data_emissao', 'danfe_status', 'id_cliente']
selected_item_cols     = ['id_item', 'item_descricao', 'descricao_tipo']
selected_cliente_cols  = ['id_cliente', 'nome_fantasia', 'id_cidade', 'estado']
selected_cidade_cols   = ['id_cidade', 'nome_cidade']

# Lean versions of each table
df_danfite_filtered  = df_danfite[selected_danfite_cols]
df_danfe_filtered    = df_danfe[selected_danfe_cols]
df_item_filtered     = df_item[selected_item_cols]
df_cliente_filtered  = df_cliente[selected_cliente_cols]
df_cidade_filtered   = df_cidade[selected_cidade_cols]

---

## 5. Consolidating the Tables (Merges)

Equivalent to Excel's VLOOKUP/XLOOKUP — progressively joins the tables using common keys.

| Merge | Key | Description |
|-------|-----|--------------|
| danfite + danfe | `id_danfe_numint` | Adds invoice header data to each item |
| + item | `id_item` | Adds product description and type |
| + customer | `id_cliente` | Adds customer name, city, and state |
| + city | `id_cidade` | Adds the full city name |


In [ ]:
# Merge 1: Invoice items (danfite) with invoice header (danfe)
df_consolidated = pd.merge(
    df_danfite_filtered,
    df_danfe_filtered,
    on='id_danfe_numint',
    how='left'  # Keeps all items; adds header data when it exists
)

# Merge 2: Add product information (item)
df_consolidated = pd.merge(
    df_consolidated,
    df_item_filtered,
    on='id_item',
    how='left'
)

# Merge 3: Add customer information
df_consolidated = pd.merge(
    df_consolidated,
    df_cliente_filtered,
    on='id_cliente',
    how='left'
)

# Merge 4: Add the city name
df_consolidated = pd.merge(
    df_consolidated,
    df_cidade_filtered,
    on='id_cidade',
    how='left'
)

df_consolidated.head()

---

## 6. Creating Year and Month Columns

Extracts year and month from the issue date and positions these columns right after the original date, making time-based filtering and analysis easier.


In [ ]:
# Ensure the column is in the correct date format
df_consolidated['data_emissao'] = pd.to_datetime(df_consolidated['data_emissao'])

# Extract year and month
df_consolidated['year'] = df_consolidated['data_emissao'].dt.year
df_consolidated['month'] = df_consolidated['data_emissao'].dt.month

# Reposition 'year' and 'month' right after 'data_emissao' for readability
cols = list(df_consolidated.columns)
cols.remove('year')
cols.remove('month')
date_position = cols.index('data_emissao')
cols.insert(date_position + 1, 'year')
cols.insert(date_position + 2, 'month')
df_consolidated = df_consolidated[cols]

df_consolidated.head()

---

## 7. Exporting to Excel

Saves the consolidated DataFrame as an `.xlsx` file in the user's Downloads folder.


In [ ]:
# Export the final result to Excel
import os
output_path = os.path.join(os.path.expanduser("~"), "Downloads", "agrosys_consolidated_report.xlsx")

df_consolidated.to_excel(output_path, index=False)
print(f"File successfully exported to: {output_path}")